# Train Custom BLIP Model on Flickr8k Dataset

This notebook fine-tunes the CustomBlipForConditionalGeneration model with additional layers on the Flickr8k dataset for image captioning task.

## 1. Import Required Libraries

In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipConfig, AdamW
from PIL import Image
import numpy as np
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Load Model and Processor

In [2]:
# Load the pre-trained model and processor
MODEL_DIRECTORY = "blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_DIRECTORY)
config = BlipConfig.from_pretrained(MODEL_DIRECTORY)

print("Model and processor loaded successfully!")

Model and processor loaded successfully!


## 3. Define Custom BLIP Model with Additional Layers

In [3]:
class CustomBlipForConditionalGeneration(BlipForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        # Define additional layers
        self.additional_layer1 = nn.Linear(768, 768)
        self.additional_layer2 = nn.Linear(768, 768)
        # Initialize additional layers
        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        encoder_outputs=None,
        past_key_values=None,
        inputs_embeds=None,
        labels=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        **kwargs,
    ):
        outputs = super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            encoder_outputs=encoder_outputs,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            **kwargs,
        )
        return outputs

# Load model
model = CustomBlipForConditionalGeneration.from_pretrained(MODEL_DIRECTORY)
model = model.to(device)
print("Custom model loaded successfully!")

Some weights of CustomBlipForConditionalGeneration were not initialized from the model checkpoint at blip-image-captioning-base and are newly initialized: ['additional_layer1.bias', 'additional_layer1.weight', 'additional_layer2.bias', 'additional_layer2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Custom model loaded successfully!


## 4. Load and Prepare Flickr8k Dataset

In [4]:
class Flickr8kDataset(Dataset):
    def __init__(self, image_dir, captions_file, processor, transform=None):
        self.image_dir = image_dir
        self.processor = processor
        self.transform = transform
        
        # Load captions from file
        self.captions = {}
        with open(captions_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    parts = line.split('\t')
                    if len(parts) == 2:
                        img_name, caption = parts
                        img_name = img_name.split('#')[0]  # Remove caption index if present
                        if img_name not in self.captions:
                            self.captions[img_name] = []
                        self.captions[img_name].append(caption)
        
        self.image_files = list(self.captions.keys())
        print(f"Loaded {len(self.image_files)} images with captions")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        
        # Load image
        if not os.path.exists(img_path):
            # Try with different extensions
            for ext in ['.jpg', '.jpeg', '.png']:
                alt_path = os.path.join(self.image_dir, img_name.replace('.jpg', ext))
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return None
        
        # Get first caption (or random one)
        caption = self.captions[img_name][0]
        
        # Process image and text
        inputs = self.processor(images=image, text=caption, return_tensors="pt", padding="max_length", truncation=True)
        
        return {
            'image_name': img_name,
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'caption': caption
        }

# Check dataset structure
dataset_dir = "Flicker8k_Dataset_test"
captions_file = "Flickr8k.token.test.txt"

if os.path.exists(dataset_dir) and os.path.exists(captions_file):
    print(f"Dataset directory found: {dataset_dir}")
    print(f"Captions file found: {captions_file}")
    print(f"Images in dataset: {len(os.listdir(dataset_dir))}")
else:
    print("⚠️  Dataset directory or captions file not found!")
    print(f"Looking for: {dataset_dir} and {captions_file}")

Dataset directory found: Flicker8k_Dataset_test
Captions file found: Flickr8k.token.test.txt
Images in dataset: 8


## 5. Create DataLoader (Optional: Only if dataset exists)

In [5]:
# Create dataset and dataloader
if os.path.exists(dataset_dir) and os.path.exists(captions_file):
    dataset = Flickr8kDataset(dataset_dir, captions_file, processor)
    batch_size = 4
    train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    print(f"DataLoader created with batch size: {batch_size}")
    print(f"Total batches: {len(train_dataloader)}")
else:
    print("Skipping DataLoader creation - dataset not found")

Loaded 8 images with captions
DataLoader created with batch size: 4
Total batches: 2


## 6. Training Setup

In [6]:
# Training parameters
num_epochs = 3
learning_rate = 5e-5
warmup_steps = 500
weight_decay = 0.01

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate total steps
if os.path.exists(dataset_dir):
    total_steps = len(train_dataloader) * num_epochs
    print(f"Total training steps: {total_steps}")
else:
    print("Dataset not found - training will not run")

Total training steps: 6


## 7. Training Loop

In [7]:
def train_epoch(model, dataloader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch + 1}")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Skip None batches
        if batch is None:
            continue
            
        # Move batch to device
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # Forward pass
        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )
        
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        avg_loss = total_loss / (batch_idx + 1)
        progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    return total_loss / len(dataloader)

# Start training only if dataset exists
if os.path.exists(dataset_dir) and os.path.exists(captions_file):
    print("Starting training...")
    for epoch in range(num_epochs):
        avg_loss = train_epoch(model, train_dataloader, optimizer, device, epoch)
        print(f"\nEpoch {epoch + 1}/{num_epochs} - Average Loss: {avg_loss:.4f}")
    print("\n✅ Training completed!")
else:
    print("⚠️  Cannot start training - Flickr8k dataset not found")

Starting training...


Epoch 1:   0%|                                        | 0/2 [00:00<?, ?it/s]


TypeError: BlipForConditionalGeneration.forward() got an unexpected keyword argument 'decoder_input_ids'

## 8. Save Trained Model

In [ ]:
# Save the model
output_dir = "blip-image-captioning-base-finetuned"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"✅ Model saved to {output_dir}")
print(f"Model files: {os.listdir(output_dir)}")

## 9. Test the Trained Model

In [ ]:
# Test inference
def test_model(model, processor, image_path, device):
    model.eval()
    
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return
    
    image = Image.open(image_path).convert('RGB')
    
    # Unconditional caption
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=50)
    caption = processor.decode(out[0], skip_special_tokens=True)
    
    print(f"Caption: {caption}")
    return caption

print("Model is ready for inference!")
print("To test, use: test_model(model, processor, 'path_to_image', device)")